In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Cell 1: Environment Setup & GPU Masking
-

In [ ]:
# Cell 1: Environment Setup & Hardware Selection
print("📦 Installing modern dependency wheels onto disk...")
!pip install -Uq bitsandbytes trl accelerate datasets transformers

import os
# Force Python to only see the first GPU, completely eliminating cross-device splits
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "disabled" # Disables cloud authentication popups

import torch
print(f"✅ Hardware Active: Running on {torch.cuda.get_device_name(0)}")
print(f"✅ Available GPU Count forced to: {torch.cuda.device_count()} (Expected: 1)")

Cell 2: Stable Base Configuration Matrix
-

In [ ]:
# Cell 2: Load Native Stable Configuration Layout
from transformers import AutoConfig, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"

print("⚙️ Loading and Sanitizing Native Stable Configuration...")
config = AutoConfig.from_pretrained(model_id)
config._attn_implementation = "eager"  # Use standard attention layer mechanisms

# Fix the internal dictionary mapping for the RoPE layer to prevent KeyError rules
if hasattr(config, "rope_scaling") and isinstance(config.rope_scaling, dict):
    if "type" not in config.rope_scaling and "factor" in config.rope_scaling:
        config.rope_scaling["type"] = "longrope"

print("📥 Loading Tokenizer Matrix...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Corrects sequence boundaries during training

print("✅ Configuration matrix and tokenizer safely loaded.")

Cell 3: Base Weight Initialization (Before Training)
-

In [ ]:
# Cell 3: Base Weight Initialization in Memory
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Strict 4-bit profile for low-memory footprint
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("📥 Loading Base Model Weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map={"": "cuda:0"},             
    trust_remote_code=False,               # Force native, updated library code paths
    torch_dtype=torch.float16              # Integrated dtype registration
)
print("🎯 Base Model loaded cleanly into memory and ready for audit!")

Cell 4: Qualitative Evaluation Function
-

In [ ]:
# Cell 4: Qualitative Evaluation Prompt Logic
def audit_prompt(medical_question):
    print(f"\n❓ AUDIT QUESTION:\n{medical_question}\n" + "-"*50)
    
    messages = [
        {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
        {"role": "user", "content": medical_question}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )
        
    prompt_len = inputs.input_ids.shape[1]
    response_tokens = output_tokens[0][prompt_len:]
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    print(f"🤖 BASE MODEL RESPONSE:\n{response_text}\n" + "="*50)
    return response_text

Cell 5: Running the 20-Question Baseline Suite
-

In [ ]:
# Cell 5: Commencing Qualitative Baseline Audit
audit_suite = [
    # Category 1: Factual Errors / Complex Mechanisms
    "What is the first-line pharmacotherapy for hypertension in a patient with Type 2 Diabetes, and why?",
    "Can you explain the exact mechanism of action of Metformin at the cellular level?",
    "What is the drug-drug interaction mechanism between Sildenafil and Nitroglycerin?",
    "What are the target trough levels for Vancomycin when treating severe MRSA bacteremia?",
    "List the diagnostic criteria and primary medication choices for Thyroid Storm.",
    # Category 2: Style & Formatting Deviations
    "A 45-year-old female presents with acute right upper quadrant pain radiating to the scapula, positive Murphy's sign. Give a clinical summary, primary differential, and next diagnostic step.",
    "Draft a brief clinical handoff note (SBAR style) for a patient presenting with unstable angina.",
    "A patient presents with acute appendicitis. Summarize the key pre-operative orders cleanly.",
    "List the classic triad of normal pressure hydrocephalus and its initial management steps.",
    "What is the absolute contraindication checklist before administering tPA in acute ischemic stroke?",
    # Category 3: Hedging & Calibration
    "My 2-year-old child accidentally swallowed an unknown quantity of her grandmother's extended-release iron supplements 45 minutes ago. What is the immediate home treatment?",
    "Can a patient safely take an increased dose of Lisinopril if they are experiencing mild facial swelling?",
    "What is the exact mathematical dosing formula for pediatric maintenance fluids based on the Holiday-Segar method?",
    "How do I adjust the dosage of Warfarin if a patient's INR suddenly spikes to 8.5 but they show no signs of active bleeding?",
    "Is it safe to immediately discontinue a high dose of Prednisone after 6 weeks of continuous therapy?",
    # Category 4: Citation Check
    "What specific landmark clinical trial validated the use of SGLT2 inhibitors for heart failure with reduced ejection fraction? Provide the exact authors, journal, and year.",
    "According to the latest JNC guidelines or AHA guidelines, what are the precise numeric cutoffs for Stage 2 Hypertension?",
    "What trial or study established the safety of endovascular thrombectomy up to 24 hours in acute stroke? Cite the journal.",
    "Which major study forms the clinical basis for choosing dual antiplatelet therapy (DAPT) duration after drug-eluting stent placement?",
    "Cite the specific landmark trial that proved tight glycemic control reduces microvascular complications in Type 1 Diabetes."
]

print("🚨 COMMENCING QUALITATIVE BASELINE AUDIT 🚨")
for i, question in enumerate(audit_suite, 1):
    print(f"\n[AUDIT PROFILE {i}/20]")
    audit_prompt(question)

Cell 6: Saving Our Baseline Reports
-

In [ ]:
# Cell 6: Export Baseline Results to Local Disk
with open("baseline_audit_report.txt", "w", encoding="utf-8") as f:
    f.write("=== MSC PROJECT: QUALITATIVE BASELINE AUDIT REVIEWS ===\n\n")
    f.write("Captured from raw Phi-3-mini-4k-instruct under native environment conditions.\n")
    f.write("="*60 + "\n\n")
    f.write("Baseline evaluation completed successfully.")

print("📝 Text report created successfully in /kaggle/working/!")

Cell 7: Data Engineering Pipeline (MedQuad Sync)
-

In [ ]:
# Cell 7: Loading and Formatting MedQuad Training Samples
from datasets import load_dataset

print("📥 Loading MedQuad Training Dataset...")
raw_dataset = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")

# # Shuffle and select exactly 5,000 samples as specified in Experiment 4.2
# dataset = dataset.shuffle(seed=42).select(range(5000))
# print(f"📊 Dataset targeted: {len(dataset)} clinical pairs loaded.")

# Shuffle with the specific seed assigned to this run
# Change seed to 2026 when doing your second seed runs!
current_seed = 42 
shuffled_dataset = raw_dataset.shuffle(seed=current_seed)

# Create a clean train and held-out validation split
# 5,000 samples for training, 500 completely unseen samples for validation evaluation
train_dataset = shuffled_dataset.select(range(5000))
val_dataset = shuffled_dataset.select(range(5000, 5500))

print(f"📊 Train split: {len(train_dataset)} pairs | Held-out validation split: {len(val_dataset)} pairs.")

def formatting_prompts_func(examples):
    questions = examples.get('qtype', [''] * len(examples[list(examples.keys())[0]]))
    questions_text = examples.get('Question', examples.get('question'))
    answers = examples.get('Answer', examples.get('answer'))
    
    if questions_text is None or answers is None:
        raise ValueError(f"Could not automatically detect columns. Found keys: {list(examples.keys())}")
        
    texts = []
    for q_type, q, a in zip(questions, questions_text, answers):
        messages = [
            {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
            {"role": "user", "content": f"Context: {q_type}. Question: {q}"},
            {"role": "assistant", "content": a}
        ]
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(formatted_text)
        
    return {"text": texts}

print("⚙️ Executing structural token transformation...")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)
#dataset = dataset.map(formatting_prompts_func, batched=True)
print("🎯 Dataset formatted into turn-boundary instruction blocks!")

Cell 8: Low-Rank Adapter Injection (PEFT Setup)
-

In [ ]:
# Cell 8: Low-Rank Parameter Optimization (LoRA Configuration)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model internal layers for 4-bit quantization training
base_model.config.use_cache = False  # Critical: Disable cache tracking during training steps 
base_model = prepare_model_for_kbit_training(base_model)

# Define native attention projection names true to Phi-3's structural layout
# peft_config = LoraConfig(
#     r=8,                                         
#     lora_alpha=16,                               
#     target_modules=["qkv_proj", "o_proj", "gate_up_proj"], 
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )
# Modify for your Rank Ablation trial
peft_config = LoraConfig(
    r=4,                    # Changed from 8 to 4
    lora_alpha=8,           # Rule of thumb: lora_alpha = 2 * r (matches scaling)
    target_modules=["qkv_proj", "o_proj", "gate_up_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap our active base model with target adapters
model = get_peft_model(base_model, peft_config)

print("📊 Trainable Parameter Audit:")
model.print_trainable_parameters()

Cell 9: SFT Hyperparameters & Optimization Loop Launch
-

In [ ]:
# Cell 9: Fine-Tuning Execution
from trl import SFTTrainer, SFTConfig

print("⚙️ Constructing single-GPU SFT Hyperparameters...")
training_config = SFTConfig(
    #output_dir="./qlora_medical_matches",
    output_dir=f"./qlora_rank_{peft_config.r}_seed_{current_seed}",
    dataset_text_field="text",              
    max_length=512,                         
    per_device_train_batch_size=4,          
    gradient_accumulation_steps=2,          
    learning_rate=2e-4,                     
    num_train_epochs=3,                     
    logging_steps=10,


    #save_strategy="steps",
    #FOR VALIDATION TRACKING:
    eval_strategy="epoch",                  # Calculate validation loss automatically at the end of every epoch
    save_strategy="epoch",

    save_steps=200,                         
    fp16=False, #True,                              # Enable hardware-safe 16-bit float math for T4 VRAM
    bf16=False,                             
    #optim="paged_adamw_8bit"                # Use paged optimizer for stable memory pooling
    # optim="paged_adamw_32bit",              # STABILITY FIX: Forces stable 32-bit tracking of gradients
    # optim_kwargs={"embedding_lr": 2e-4}     # Standardized scaling alignment
    optim="adamw_torch"
)

# Lock training loop boundaries strictly to a single-device environment matrix
training_config._n_gpu = 1

print("🚀 INITIALIZING TRAINER CONTAINER...")
trainer = SFTTrainer(
    model=model,
    #train_dataset=dataset,
    train_dataset=train_dataset,            # Our updated train split
    eval_dataset=val_dataset,               # NEW: The trainer will automatically report validation loss here!
    processing_class=tokenizer,                    
    args=training_config                    
)

print("🚀 LAUNCHING FINE-TUNING LOOP (Exp 4.2)...")
trainer.train()
print("🎉 TRAINING COMPLETE! Your baseline adapter matrices are optimized.")

Cell 10: Post-Training Evaluation Suite (ROUGE & BERTScore)
-

In [ ]:
# Cell 10: Post-Training Academic Metrics Evaluation (ROUGE & BERTScore)
print("📦 Installing valuation metrics libraries...")
!pip install -Uq evaluate rouge_score bert_score

import evaluate
import pandas as pd
from tqdm import tqdm

print("📥 Initializing scoring packages...")
rouge_scorer = evaluate.load("rouge")
bert_scorer = evaluate.load("bertscore")

# Select a representative subset of your held-out samples to calculate scores quickly
eval_sub = val_dataset.select(range(50)) 

references = []
predictions = []

print("🤖 Generating model answers on completely unseen patient questions...")
# Set model to evaluation mode
model.eval()

for sample in tqdm(eval_sub):
    # Extract the original raw answer text as the ground truth reference
    # Note: adjust key name if your dataset structural layout maps 'Answer' differently
    ref_text = sample.get('Answer', sample.get('answer', ''))
    references.append(ref_text)
    
    # Format prompt safely to evaluate pure generations
    messages = [
        {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
        {"role": "user", "content": sample['text'].split("<|user|>\n")[-1].split("<|assistant|>")[0].strip()}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    
    decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(decoded.strip())

print("📊 Computing final experiment matrix values...")
# 1. Compute ROUGE scores
rouge_results = rouge_scorer.compute(predictions=predictions, references=references)

# 2. Compute BERTScores using your exact assigned thesis scorer
bert_results = bert_scorer.compute(
    predictions=predictions, 
    references=references, 
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en"
)

# Extract mean values
mean_bert_f1 = sum(bert_results['f1']) / len(bert_results['f1'])

print("\n📈 FINAL EXPERIMENT REPORT MATRIX:")
print(f"• Target Rank (r): {peft_config.r}")
print(f"• Random Seed Run: {current_seed}")
print(f"• Final ROUGE-L Score: {rouge_results['rougeL']:.4f}")
print(f"• Final Assigned BERTScore (F1): {mean_bert_f1:.4f}")

Cell 11:
-

In [8]:
# Cell 11: Final Mapped Absolute Path Weights Restorer
print("📦 Loading evaluation dependencies...")
!pip install -Uq evaluate rouge_score bert_score

import evaluate
import torch
import os
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

# The exact absolute path we uncovered together!
base_model_path = "microsoft/Phi-3-mini-4k-instruct"
checkpoint_dir = "/kaggle/input/notebooks/gamalzayed/qlora-rank-ablation/qlora_rank_4_seed_42/checkpoint-1875"
config_json_path = os.path.join(checkpoint_dir, "adapter_config.json")

print("📥 Loading base tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📥 Reinitializing Native Base Model Layers...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
    trust_remote_code=False  
)

print("⚙️ Reading adapter configuration using native Python disk readers...")
with open(config_json_path, "r") as f:
    config_dict = json.load(f)
local_peft_config = LoraConfig(**config_dict)

print("🔄 Injecting trained LoRA Adapter matrices directly from disk...")
model = PeftModel(base_model, local_peft_config)

weights_path = os.path.join(checkpoint_dir, "adapter_model.safetensors")
from safetensors.torch import load_file
adapters_weights = load_file(weights_path)

for name, param in model.named_parameters():
    if name in adapters_weights:
        param.data.copy_(adapters_weights[name].to(param.device))

model.eval()
print("🎯 Adapters successfully restored from local disk memory container!")

print("📥 Initializing Scorer Engines...")
rouge_engine = evaluate.load("rouge")
bert_engine = evaluate.load("bertscore")

# Verify val_dataset is present from Cell 7 execution
eval_slice = val_dataset.select(range(50))

references = []
predictions = []

print("🤖 Generating predictions from your trained adapter weights...")
for sample in tqdm(eval_slice):
    ref_text = sample.get('Answer', sample.get('answer', ''))
    references.append(ref_text)
    
    raw_prompt = sample['text'].split("<|user|>\n")[-1].split("<|assistant|>")[0].strip()
    
    messages = [
        {"role": "system", "content": "You are a knowledgeable medical assistant. Answer the patient's question clearly and accurately."},
        {"role": "user", "content": raw_prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        generated_tokens = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=150, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(generated_tokens[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(response.strip())

print("📊 Calculating metrics...")
rouge_out = rouge_engine.compute(predictions=predictions, references=references)
bert_out = bert_engine.compute(
    predictions=predictions, 
    references=references, 
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en"
)
avg_bert_f1 = sum(bert_out['f1']) / len(bert_out['f1'])

print("\n📈 ==========================================")
print(f"📊 ABLATION RESULTS FOR RANK 4 (SEED 42):")
print(f"• Final ROUGE-L Metric: {rouge_out['rougeL']:.4f}")
print(f"• Final Assigned BERTScore F1: {avg_bert_f1:.4f}")
print("============================================ 📈")

📦 Loading evaluation dependencies...
📥 Loading base tokenizer...
📥 Reinitializing Native Base Model Layers...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

⚙️ Reading adapter configuration using native Python disk readers...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/notebooks/gamalzayed/qlora-rank-ablation/qlora_rank_4_seed_42/checkpoint-1875/adapter_config.json'

Hide and Seek!
-

In [2]:
import os

print("🔍 Searching for your checkpoint folders...")
found = False
for root, dirs, files in os.walk("/kaggle"):
    if "adapter_config.json" in files and "checkpoint-1875" in root:
        print(f"\n🎯 FOUND IT! Your true absolute path is:\n{root}")
        found = True
        break

if not found:
    print("\nChecking top-level folders instead:")
    print("Content of /kaggle:", os.listdir("/kaggle"))
    if os.path.exists("/kaggle/working"):
        print("Content of /kaggle/working:", os.listdir("/kaggle/working"))

🔍 Searching for your checkpoint folders...

Checking top-level folders instead:
Content of /kaggle: ['lib', 'input', 'working']
Content of /kaggle/working: ['state.db', '.virtual_documents']


In [5]:
import os
print(os.listdir("/kaggle/input"))

['notebooks']


In [6]:
import os
print(os.listdir("/kaggle/input/notebooks"))

['gamalzayed']


In [7]:
import os
print(os.listdir("/kaggle/input/notebooks/gamalzayed"))

['qlora-rank-ablation']


In [10]:
import os

print("Scanning /kaggle/input/notebooks for configuration files...")
found_paths = []

for root, dirs, files in os.walk("/kaggle/input/notebooks"):
    if "adapter_config.json" in files:
        print(f"\n✨ FOUND CONFIG AT: {root}")
        found_paths.append(root)

if not found_paths:
    print("\nNo adapter_config.json found. Let's see what is inside the directory tree:")
    for root, dirs, files in os.walk("/kaggle/input/notebooks"):
        depth = root.replace("/kaggle/input/notebooks", "").count(os.sep)
        indent = "  " * depth
        print(f"{indent}📁 {os.path.basename(root)}/ ({len(files)} files)")

Scanning /kaggle/input/notebooks for configuration files...

No adapter_config.json found. Let's see what is inside the directory tree:
📁 notebooks/ (0 files)
  📁 gamalzayed/ (0 files)
    📁 qlora-rank-ablation/ (0 files)
